# Traffic Signal Optimization with RL — Experiments

Single notebook driving the whole experimental pipeline. Run top to bottom.

**Three phases:**
1. **Baselines** — fixed-time and max-pressure across all scenarios
2. **Single-intersection RL** — PPO and DQN on cologne1, multi-seed + reward ablation
3. **Multi-intersection MARL** — shared-policy IPPO on cologne3 and cologne8

Every training cell is restartable: if a model file already exists on disk, training is skipped.

**Approximate runtime** (one full pass, all seeds):
- WSL2 + libsumo: ~15 hours
- Windows + TraCI: ~150 hours

See README.md for the WSL2 setup that gets you the 10x speedup.

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from ppo import train as train_ppo
from dqn import train as train_dqn
from ippo import train as train_ippo
import fixed_time
import max_pressure
from utils.evaluate import evaluate, evaluate_marl

from stable_baselines3 import PPO, DQN

# Configuration
TRAINING_SEEDS = [42, 43, 44, 45, 46]
ABLATION_SEEDS = [42, 43, 44]
MARL_SEEDS     = [42, 43, 44]
EVAL_SEEDS     = [100, 200, 300, 400, 500]
ALT_REWARDS    = ["pressure", "queue", "average-speed"]

Path("results").mkdir(exist_ok=True)
print("Setup OK")

## Phase 1 — Baselines

No training. Run fixed-time and max-pressure on every scenario, evaluate on 5 seeds each.

In [ ]:
baseline_frames = []
for scenario in ["cologne1", "cologne3", "cologne8"]:
    print(f"\n=== {scenario} ===")
    for name, runner in [("fixed_time", fixed_time.run), ("max_pressure", max_pressure.run)]:
        df = runner(scenario=scenario, eval_seeds=EVAL_SEEDS)
        df["scenario"] = scenario
        df["algo"] = name
        baseline_frames.append(df)

baselines = pd.concat(baseline_frames, ignore_index=True)
baselines.to_csv("results/baselines.csv", index=False)
print(f"\nSaved {len(baselines)} baseline rows -> results/baselines.csv")
baselines.groupby(["scenario", "algo"]).agg(
    wait=("mean_wait", "mean"),
    speed=("mean_speed", "mean"),
    stopped=("mean_stopped", "mean"),
).round(2)

## Phase 2 — Single-intersection RL on cologne1

PPO and DQN across 5 training seeds, then PPO reward ablation across 3 seeds x 3 alternate rewards.

### 2.1 Train PPO and DQN, 5 seeds each

In [ ]:
for seed in TRAINING_SEEDS:
    train_ppo(seed=seed)
    train_dqn(seed=seed)

### 2.2 Reward ablation (PPO only)

In [ ]:
for reward in ALT_REWARDS:
    for seed in ABLATION_SEEDS:
        train_ppo(seed=seed, reward_fn=reward)

### 2.3 Evaluate every trained model

In [ ]:
def _load_eval(algo, seed, reward_fn="diff-waiting-time"):
    tag = f"seed{seed}"
    if reward_fn != "diff-waiting-time":
        tag += f"_{reward_fn}"
    model_path = Path(f"checkpoints/cologne1/{algo}/{tag}/final.zip")
    if not model_path.exists():
        print(f"  missing: {model_path}")
        return None
    print(f"Evaluating {algo} {tag}")
    cls = PPO if algo == "ppo" else DQN
    model = cls.load(str(model_path))
    df = evaluate(model, eval_seeds=EVAL_SEEDS, reward_fn=reward_fn, scenario="cologne1")
    df["algo"] = algo
    df["train_seed"] = seed
    df["reward_fn"] = reward_fn
    df["scenario"] = "cologne1"
    return df


cologne1_frames = []
for seed in TRAINING_SEEDS:
    for algo in ["ppo", "dqn"]:
        df = _load_eval(algo, seed)
        if df is not None: cologne1_frames.append(df)

for reward in ALT_REWARDS:
    for seed in ABLATION_SEEDS:
        df = _load_eval("ppo", seed, reward_fn=reward)
        if df is not None: cologne1_frames.append(df)

cologne1_results = pd.concat(cologne1_frames, ignore_index=True)
cologne1_results.to_csv("results/cologne1_rl.csv", index=False)
print(f"Saved {len(cologne1_results)} rows -> results/cologne1_rl.csv")

### 2.4 Summary table — PPO vs DQN vs reward ablation

In [ ]:
cologne1_summary = cologne1_results.groupby(["algo", "reward_fn"]).agg(
    n_seeds=("train_seed", "nunique"),
    wait_mean=("mean_wait", "mean"),
    wait_std=("mean_wait", "std"),
    speed_mean=("mean_speed", "mean"),
    speed_std=("mean_speed", "std"),
    stopped_mean=("mean_stopped", "mean"),
    stopped_std=("mean_stopped", "std"),
).round(3)
cologne1_summary

### 2.5 PPO vs DQN comparison plot (default reward)

In [ ]:
comp = cologne1_results[cologne1_results["reward_fn"] == "diff-waiting-time"]
agg = comp.groupby("algo").agg(
    wait_mean=("mean_wait", "mean"), wait_std=("mean_wait", "std"),
    speed_mean=("mean_speed", "mean"), speed_std=("mean_speed", "std"),
    stopped_mean=("mean_stopped", "mean"), stopped_std=("mean_stopped", "std"),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = {"ppo": "#0ea5e9", "dqn": "#e8772e"}
for ax, (m, s, label) in zip(axes, [
    ("wait_mean", "wait_std", "Mean wait (s)"),
    ("speed_mean", "speed_std", "Mean speed (m/s)"),
    ("stopped_mean", "stopped_std", "Avg stopped vehicles"),
]):
    ax.bar(agg["algo"].str.upper(), agg[m], yerr=agg[s],
           color=[colors[a] for a in agg["algo"]], capsize=8,
           edgecolor="black", linewidth=0.5)
    for i, (mi, si) in enumerate(zip(agg[m], agg[s])):
        ax.text(i, mi + si + max(agg[m]) * 0.03, f"{mi:.2f}",
                ha="center", va="bottom", fontsize=10)
    ax.set_ylabel(label)
    ax.grid(axis="y", alpha=0.3)
fig.suptitle("PPO vs DQN on cologne1 (5 training seeds, mean +/- std)", fontweight="bold")
fig.tight_layout()
plt.savefig("results/cologne1_ppo_vs_dqn.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.6 Reward ablation plot

In [ ]:
ppo_only = cologne1_results[cologne1_results["algo"] == "ppo"]
abl = ppo_only.groupby("reward_fn").agg(
    wait_mean=("mean_wait", "mean"), wait_std=("mean_wait", "std"),
    p95_mean=("p95_wait", "mean"),   p95_std=("p95_wait", "std"),
).reset_index().sort_values("wait_mean")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (m, s, label) in zip(axes, [
    ("wait_mean", "wait_std", "Mean waiting time (s) - lower is better"),
    ("p95_mean",  "p95_std",  "p95 waiting time (s) - tail / fairness"),
]):
    ax.bar(abl["reward_fn"], abl[m], yerr=abl[s], color="#0ea5e9",
           capsize=8, edgecolor="black", linewidth=0.5)
    for i, (mi, si) in enumerate(zip(abl[m], abl[s])):
        ax.text(i, mi + si + max(abl[m]) * 0.03, f"{mi:.2f}",
                ha="center", va="bottom", fontsize=10)
    ax.set_ylabel(label)
    ax.set_xticklabels(abl["reward_fn"], rotation=15, ha="right")
    ax.grid(axis="y", alpha=0.3)
fig.suptitle("PPO reward-function ablation on cologne1", fontweight="bold")
fig.tight_layout()
plt.savefig("results/cologne1_reward_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

## Phase 3 — Multi-intersection MARL

Shared-policy IPPO on cologne3 (3 signals) and cologne8 (8 signals).
With N intersections, each SB3 timestep = 1/N SUMO steps -- cologne8 needs more timesteps.

### 3.1 Train IPPO on every (scenario, seed)

In [ ]:
marl_config = {
    "cologne3": 500_000,
    "cologne8": 800_000,
}

for scenario, timesteps in marl_config.items():
    for seed in MARL_SEEDS:
        train_ippo(seed=seed, scenario=scenario, timesteps=timesteps)

### 3.2 Evaluate every IPPO model

In [ ]:
marl_frames = []
for scenario in marl_config:
    for seed in MARL_SEEDS:
        model_path = Path(f"checkpoints/{scenario}/ippo/seed{seed}/final.zip")
        if not model_path.exists():
            print(f"  missing: {model_path}")
            continue
        print(f"Evaluating ippo {scenario}/seed{seed}")
        model = PPO.load(str(model_path))
        df = evaluate_marl(model, eval_seeds=EVAL_SEEDS, scenario=scenario)
        df["algo"] = "ippo"
        df["train_seed"] = seed
        df["scenario"] = scenario
        marl_frames.append(df)

marl_results = pd.concat(marl_frames, ignore_index=True)
marl_results.to_csv("results/marl.csv", index=False)
print(f"Saved {len(marl_results)} rows -> results/marl.csv")

### 3.3 MARL vs baselines plot

In [ ]:
marl_scenarios = list(marl_config.keys())
b = baselines[baselines["scenario"].isin(marl_scenarios)][
    ["scenario", "algo", "mean_wait", "mean_speed", "mean_stopped", "p95_wait"]
].copy()
b["train_seed"] = -1
m = marl_results[
    ["scenario", "algo", "train_seed", "mean_wait", "mean_speed", "mean_stopped", "p95_wait"]
].copy()
combined = pd.concat([b, m], ignore_index=True)

agg = combined.groupby(["scenario", "algo"]).agg(
    wait_mean=("mean_wait", "mean"), wait_std=("mean_wait", "std"),
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
scenarios = sorted(agg["scenario"].unique())
algos = ["fixed_time", "max_pressure", "ippo"]
colors = {"fixed_time": "#94a3b8", "max_pressure": "#a855f7", "ippo": "#0ea5e9"}

import numpy as np
x = np.arange(len(scenarios))
width = 0.27
for i, algo in enumerate(algos):
    rows = agg[agg["algo"] == algo].set_index("scenario").reindex(scenarios).reset_index()
    means = rows["wait_mean"].fillna(0).values
    stds = rows["wait_std"].fillna(0).values
    ax.bar(x + (i - 1) * width, means, width, yerr=stds, label=algo,
           color=colors[algo], capsize=4, edgecolor="black", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(scenarios)
ax.set_ylabel("Mean waiting time (s)")
ax.set_title("Multi-intersection: IPPO vs classical baselines", fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.savefig("results/marl_vs_baselines.png", dpi=150, bbox_inches="tight")
plt.show()
agg

## Final results -- combined table

One table covering every (scenario, algorithm) pair, with std across training seeds where applicable.

In [ ]:
b = baselines.copy()
b["train_seed"] = -1
b["reward_fn"] = "n/a"

c1 = cologne1_results[cologne1_results["reward_fn"] == "diff-waiting-time"].copy()
final = pd.concat([
    b[["scenario", "algo", "train_seed", "reward_fn", "mean_wait", "mean_speed", "mean_stopped", "p95_wait"]],
    c1[["scenario", "algo", "train_seed", "reward_fn", "mean_wait", "mean_speed", "mean_stopped", "p95_wait"]],
    marl_results.assign(reward_fn="diff-waiting-time")[
        ["scenario", "algo", "train_seed", "reward_fn", "mean_wait", "mean_speed", "mean_stopped", "p95_wait"]
    ],
], ignore_index=True)

final_summary = final.groupby(["scenario", "algo"]).agg(
    n_seeds=("train_seed", "nunique"),
    wait_mean=("mean_wait", "mean"),  wait_std=("mean_wait", "std"),
    speed_mean=("mean_speed", "mean"), speed_std=("mean_speed", "std"),
    stopped_mean=("mean_stopped", "mean"), stopped_std=("mean_stopped", "std"),
    p95_mean=("p95_wait", "mean"), p95_std=("p95_wait", "std"),
).round(3)

final_summary.to_csv("results/final_summary.csv")
print("Saved -> results/final_summary.csv")
final_summary